# MINJA 最小机制复现（CPU / 离线）

目标：演示恶意交互记录进入长期记忆后，如何因 Top-k 相似度检索影响后续查询。这里使用纯 Python TF-IDF 和确定性 toy agent；LLM 生成桥接推理的阶段由测试夹具模拟，因此结果不能与论文 GPT 实验数值直接比较。

## 1. 加载实验代码与合成数据

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "data").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "code"))

from run_minja_toy import load_experiment, run_experiment

config = load_experiment(ROOT / "data" / "minja_toy_records.json")
len(config["benign_records"]), len(config["scenarios"])

数据包含 24 条良性记忆和 3 个完全虚构的 victim-target 场景。每个场景有三轮攻击查询，文本逐步缩短并接近受害查询。

In [ ]:
scenario = config["scenarios"][0]
print("Victim query:", scenario["victim_query"])
for index, query in enumerate(scenario["attack_queries"], 1):
    print(f"Round {index}: {query}")

## 2. 运行基线、注入和渐进缩短实验

In [ ]:
rows, summary = run_experiment(config, top_k=3, seed=42)

for item in summary["scenarios"]:
    print("\nScenario:", item["scenario"])
    print("Baseline:", item["baseline_answer"])
    print("After injection:", item["final_answer"])
    print("ISR / ASR / utility:", item["simplified_isr"], item["simplified_asr"], item["utility_retention"])

## 3. 查看每轮检索指标

In [ ]:
header = ("scenario", "round", "poison_similarity", "poison_rank", "in_top_k", "attack_success")
print(" | ".join(header))
for row in rows:
    print(" | ".join(str(row[key]) for key in header))

## 4. 应如何解释结果

- 基线准确率为 1，表示注入前 3 个 victim query 都检索到正确良性记录。
- 最后一轮恶意记录与 victim query 最接近并排名 Top-1，toy agent 因而采用错误目标答案。
- 无关查询的 Top-1 保持率用于近似正常效用；它不是论文完整 UD。
- 真正 MINJA 的关键难点是仅用查询诱导 LLM 自己生成目标桥接轨迹。本实验没有真实 LLM，所以只复现后半段的写入、检索和影响链路。